In [1]:
# Performing 70-30 train-validation split within the entire region
# Stations within 1 pixel distance (diagonal) of ERA5L (coarsest data) is avoided from split using DBSCAN

In [2]:
import pandas as pd
import numpy as np

import xarray as xr 
import rasterio as rio
import rioxarray as rxr

import os
from sklearn.cluster import DBSCAN

output_dir = r"../Data_Temp/Processed/Split_stations"
os.makedirs(output_dir, exist_ok= True)

stn_tmax = pd.read_csv(r"../Data_Temp/Processed/Initial_accuracy/unique_stn_tmax.csv")
stn_tmin = pd.read_csv(r"../Data_Temp/Processed/Initial_accuracy/unique_stn_tmin.csv")

kms_per_radian = 6371.0088
search_distance_km = 17
epsilon = search_distance_km / kms_per_radian

In [3]:
# Add a new column of db cluster

def add_db_cluster(df_stn):
    coordinates = np.radians(df_stn[['lat', 'long']].to_numpy())
    db = DBSCAN(eps = epsilon, 
                        min_samples = 1, 
                        algorithm='ball_tree', 
                        metric='haversine').fit(coordinates)
    df_stn['db_cluster'] = db.labels_
    noise_count = df_stn[df_stn['db_cluster'] == -1].shape[0]
    print("Number of noise stations:", noise_count)

    return df_stn

tmax_stn_db = add_db_cluster(stn_tmax)
tmin_stn_db = add_db_cluster(stn_tmin)

Number of noise stations: 0
Number of noise stations: 0


In [5]:
# Split stations by dbcluster value into train and test


split_fraction = 0.7

def split_stations(df_stn_db: pd.DataFrame(), t_variable: str):
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()
    df_stn_db = df_stn_db.set_index('db_cluster')
    db_clusters = df_stn_db.index.unique()
    
    train_clusters = pd.Series(db_clusters).sample(frac=split_fraction, random_state = 64)
    train_lv = df_stn_db.loc[train_clusters].reset_index()
    test_lv = df_stn_db.drop(train_clusters).reset_index()

    train_df = pd.concat([train_df, train_lv], ignore_index = True)
    test_df = pd.concat([test_df, test_lv], ignore_index=True)

    train_df.to_csv(os.path.join(output_dir, f'train_station_{t_variable}.csv'), index = False)
    test_df.to_csv(os.path.join(output_dir, f'test_station_{t_variable}.csv'), index = False)

    print("Files saved to:", os.path.join(output_dir, f'test_station_{t_variable}.csv'))

    return None

split_stations(tmax_stn_db, "tmax")
split_stations(tmin_stn_db, "tmin")

Files saved to: ../Data_Temp/Processed/Split_stations\test_station_tmax.csv
Files saved to: ../Data_Temp/Processed/Split_stations\test_station_tmin.csv
